# 2. Neural-network study

**CSI 4106 Introduction to Artificial Intelligence**  
*Assignment 3: Designing and Diagnosing Neural Networks*  
*Due: November 16, 2026, at 11 PM*

# Identification

Name:  
Student number: Report title:

# 1. Data and objective

The objective is to predict whether an autonomous underwater vehicle
returns before completing its planned survey. The positive class is
`mission_interrupted=1`.

The following cell supplies the imports, feature groups, fixed data
partitions, and an unfitted preprocessing factory used throughout the
assignment. Do not change the supplied feature groups, split settings,
or DataFrame and Series names. You remain responsible for fitting and
applying the preprocessor at the correct stages and for preparing the
NumPy arrays used by Keras.

In [41]:
from pathlib import Path
import math
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from tensorflow import keras


COMPLETE_URL = (
    "https://raw.githubusercontent.com/turcotte/csi4106-f26/main/"
    "assignments-data/a2/asrc_missions_complete.csv"
)
HIDDEN_URL = (
    "https://raw.githubusercontent.com/turcotte/csi4106-f26/main/"
    "assignments-data/a2/asrc_missions_hidden.csv"
)

COMPLETE_LOCAL_PATH = Path("asrc_missions_complete.csv")
HIDDEN_LOCAL_PATH = Path("asrc_missions_hidden.csv")

TARGET = "mission_interrupted"
NUMERIC_FEATURES = [
    "planned_depth_m",
    "descent_current_speed_m_s",
    "water_temperature_c",
    "sonar_noise_db",
    "navigation_drift_m",
    "battery_health_pct",
    "thruster_vibration_mm_s",
    "missions_since_service",
]
NOMINAL_FEATURES = ["vehicle_class", "seabed_terrain"]
ORDINAL_FEATURES = ["maintenance_grade"]
BINARY_FEATURES = ["firmware_current"]
FEATURES = (
    NUMERIC_FEATURES
    + NOMINAL_FEATURES
    + ORDINAL_FEATURES
    + BINARY_FEATURES
)
EXPECTED_COLUMNS = FEATURES + [TARGET]


def make_preprocessor():
    """Return a new, unfitted copy of the required preprocessor."""
    return ColumnTransformer(
        transformers=[
            ("numeric", StandardScaler(), NUMERIC_FEATURES),
            (
                "nominal",
                OneHotEncoder(handle_unknown="ignore", sparse_output=False),
                NOMINAL_FEATURES,
            ),
            (
                "ordinal",
                OrdinalEncoder(
                    categories=[["poor", "fair", "good", "excellent"]]
                ),
                ORDINAL_FEATURES,
            ),
            ("binary", "passthrough", BINARY_FEATURES),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )


complete_source = COMPLETE_LOCAL_PATH if COMPLETE_LOCAL_PATH.exists() else COMPLETE_URL
missions = pd.read_csv(complete_source)

if list(missions.columns) != EXPECTED_COLUMNS:
    raise ValueError(
        "The complete data do not have the expected columns in the expected order."
    )
if len(missions) != 6371:
    raise ValueError(f"Expected 6,371 labelled missions, got {len(missions):,}.")
if missions.isna().any().any():
    raise ValueError("The complete data contain missing values.")
if not set(missions[TARGET].unique()).issubset({0, 1}):
    raise ValueError("The response must contain only 0 and 1.")

X_labelled = missions[FEATURES].copy()
y_labelled = missions[TARGET].copy()

X_development, X_test, y_development, y_test = train_test_split(
    X_labelled,
    y_labelled,
    test_size=0.20,
    random_state=4106,
    stratify=y_labelled,
)

X_neural_train, X_validation, y_neural_train, y_validation = train_test_split(
    X_development,
    y_development,
    test_size=0.20,
    random_state=4106,
    stratify=y_development,
)

print(f"Loaded labelled data from: {complete_source}")
print(f"Labelled data: {missions.shape}")
print(
    "Partitions: "
    f"neural training={len(X_neural_train)}, "
    f"validation={len(X_validation)}, "
    f"development={len(X_development)}, "
    f"test={len(X_test)}"
)

Loaded labelled data from: https://raw.githubusercontent.com/turcotte/csi4106-f26/main/assignments-data/a2/asrc_missions_complete.csv
Labelled data: (6371, 13)
Partitions: neural training=4076, validation=1020, development=5096, test=1275


## 2.1 Prepare and validate the neural-network inputs

Summarize the complete, development, neural-training, and validation
data as specified in Task 1. Report only the number of reserved test
observations before the final evaluation. Instantiate the supplied
preprocessor, fit it only on the neural-network training predictors, and
transform the validation predictors. Convert predictors and targets to
NumPy arrays suitable for Keras.

Validate and report the array shapes and data types. Confirm that all
values are finite, targets contain only `0` and `1`, and each predictor
array is aligned with its target array. Report the transformed input
width and explain the preprocessing and leakage boundary.

In [42]:
print(f"Shape: {missions.shape}")
print(f"First five rows: \n{missions.head()}")
print(f"complete interrupted count: \n{missions[TARGET].value_counts()}")
print(f"complete interrupted proportion: \n{missions[TARGET].value_counts(normalize=True) * 100}")
print(f"development interrupted count: \n{y_development.value_counts()}")
print(f"development interrupted proportion: \n{y_development.value_counts(normalize=True) * 100}")
print(f"neural-network interrupted count: \n{y_neural_train.value_counts()}")
print(f"neural-network interrupted proportion: \n{y_neural_train.value_counts(normalize=True) * 100}")
print(f"validation interrupted count: \n{y_validation.value_counts()}")
print(f"validation interrupted proportion: \n{y_validation.value_counts(normalize=True) * 100}")

print(f"test number of observations: {len(X_test)}")

processor = make_preprocessor()
neural_processed = processor.fit_transform(X_neural_train)
neural_validation = processor.transform(X_validation)

X_train_np = np.array(neural_processed, dtype=np.float32)
X_validation_np = np.array(neural_validation, dtype=np.float32)
y_train_np = y_neural_train.to_numpy(dtype=np.float32)
y_validation_np = y_validation.to_numpy(dtype=np.float32)

print(f"X_train_np shape: {X_train_np.shape}, X_train_np data type: {X_train_np.dtype}")
print(f"y_train_np shape: {y_train_np.shape}, y_train_np data type: {y_train_np.dtype}")

print(f"X_validation_np shape: {X_validation_np.shape}, X_validation_np data type: {X_validation_np.dtype}")
print(f"y_validation_np shape: {y_validation_np.shape}, y_validation_np data type: {y_validation_np.dtype}")

print(f"X_train_np has finite values: {np.isfinite(X_train_np).all()}")
print(f"y_train_np has finite values: {np.isfinite(y_train_np).all()}")

print(f"X_validation_np has finite values: {np.isfinite(X_validation_np).all()}")
print(f"y_validation_np has finite values: {np.isfinite(y_validation_np).all()}")

print(f"training targets contain only 0 and 1: {np.isin(y_train_np, [0, 1]).all()}")
print(f"validation targets contain only 0 and 1: {np.isin(y_validation_np, [0, 1]).all()}")

print(f"training arrays aligned: {X_train_np.shape[0] == y_train_np.shape[0]}")
print(f"validation arrays aligned: {X_validation_np.shape[0] == y_validation_np.shape[0]}")

print(f"input width: {X_train_np.shape[1]}")

Shape: (6371, 13)
First five rows: 
   planned_depth_m  descent_current_speed_m_s  water_temperature_c  \
0           1400.0                       0.55                  3.3   
1           2700.0                       0.68                  3.0   
2           2870.0                       0.15                  3.6   
3           2840.0                       0.39                  2.5   
4           2430.0                       0.34                  2.8   

   sonar_noise_db  navigation_drift_m  battery_health_pct  \
0            66.0                10.4                87.0   
1            78.5                49.9                83.0   
2            78.7                 5.8                92.0   
3            77.3                54.6                89.0   
4            77.8                 8.7                89.0   

   thruster_vibration_mm_s  missions_since_service vehicle_class  \
0                      3.1                      14       compact   
1                      3.9              

**Input validation and preprocessing rationale:**

Validation is done by checking the number of observations, number of input features, input data type, all values being finite, targets containing only 0 and 1and that predictors and targets are aligned.

preprocessing applied to each predictor group: standard scaler for numeric so that variables with large numerical ranges don't dominate, one-hot encoding for nominal so that no order is created with categorical values, ordinal encoding for ordinal so that order is preserved with categorical values and passthrough for binary since further encoding is unnecessary. The preprocessor must be fitted on neural-network training rows rather than on the complete development partition to prevent data leakage.

## 2.2 Connect logistic regression to a small neural network

Train the required no-hidden-layer and 8-unit models. Calculate and
verify their parameter counts, plot loss and positive-class F1 at
threshold `0.5`, report metrics recomputed after restoring the
lowest-validation-loss weights, diagnose their behaviour, and explain
the connection to logistic regression.

In [56]:
from tensorflow.keras.callbacks import ModelCheckpoint

keras.utils.set_random_seed(4106)
CLASSIFICATION_THRESHOLD = 0.5
no_hidden_model = keras.Sequential(
    [keras.Input(shape=(X_train_np.shape[1],)), keras.layers.Dense(1, activation="sigmoid")],
    name="no_hidden_layer",
)
no_hidden_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=[keras.metrics.BinaryAccuracy(threshold=CLASSIFICATION_THRESHOLD)],
)

# no_hidden_parameter_count = no_hidden_model.count_params()
# assert no_hidden_parameter_count == X_train_np.shape[1] + 1
no_hidden_checkpoint_path = Path("no_hidden_layer_best.weights.h5")
no_hidden_checkpoint = ModelCheckpoint(
    filepath=no_hidden_checkpoint_path,
    monitor="val_loss",
    mode="min",
    save_best_only=True,
    save_weights_only=True,
    verbose=0
)

no_hidden_history = no_hidden_model.fit(
    X_train_np,
    y_train_np,
    validation_data=(X_validation_np, y_validation_np),
    epochs=100,
    batch_size=64,
    callbacks=[no_hidden_checkpoint],
    verbose=0
)

best_epoch = int(np.argmin(no_hidden_history.history["val_loss"])) + 1
no_hidden_model.load_weights(no_hidden_checkpoint_path)
best_validation_loss, best_validation_accuracy = no_hidden_model.evaluate(
    X_validation_np, y_validation_np, verbose=0
)
validation_probabilities = no_hidden_model.predict(X_validation_np, verbose=0).ravel()
validation_predictions = (
    validation_probabilities >= CLASSIFICATION_THRESHOLD
).astype(int)

print(f"Architecture: {X_train_np.shape[1]} inputs -> 1 sigmoid output, no hidden layer")
# print(f"Trainable parameters: {no_hidden_parameter_count}")
print(f"Completed epochs: {len(no_hidden_history.epoch)}")
print(f"Lowest validation loss epoch: {best_epoch}")
print(f"Saved best weights: {no_hidden_checkpoint_path}")
print(f"Validation loss at best checkpoint: {best_validation_loss:.4f}")
print(f"Validation accuracy at threshold {CLASSIFICATION_THRESHOLD}: {best_validation_accuracy:.4f}")
print(f"Validation precision: {precision_score(y_validation_np, validation_predictions, zero_division=0):.4f}")
print(f"Validation recall: {recall_score(y_validation_np, validation_predictions, zero_division=0):.4f}")
print(f"Validation F1: {f1_score(y_validation_np, validation_predictions, zero_division=0):.4f}")
print(f"Validation ROC AUC: {roc_auc_score(y_validation_np, validation_probabilities):.4f}")

Architecture: 17 inputs -> 1 sigmoid output, no hidden layer
Completed epochs: 100
Lowest validation loss epoch: 100
Saved best weights: no_hidden_layer_best.weights.h5
Validation loss at best checkpoint: 0.4994
Validation accuracy at threshold 0.5: 0.7863
Validation precision: 0.7280
Validation recall: 0.5482
Validation F1: 0.6254
Validation ROC AUC: 0.7862


**Results and interpretation:**

## 2.3 Study depth and capacity

Train and compare the three approximately matched-parameter
architectures. Calculate and verify their parameter counts, report their
results, plot comparable curves, and interpret the effect of depth. Then
calculate the parameter counts for the three equal-hidden-unit
architectures and explain why that comparison is not controlled. Do not
train the additional `192` and `96, 96` models.

In [44]:
# Your code

**Depth, capacity, and protocol critique:**

## 2.4 Evaluate early stopping

Apply the required early-stopping configuration to a fresh `64, 64, 64`
ReLU model. Compare it with the unregularized reference using completed
epochs, best epoch, elapsed time, curves, and validation metrics.

In [45]:
# Your code

**Early-stopping analysis:**

## 2.5 Compare activation functions

Train the required `tanh` model and compare it with the ReLU reference
while keeping every other setting fixed.

In [46]:
# Your code

**Controlled comparison and interpretation:**

## 2.6 Diagnose batch-size effects

Compare the required batch sizes, reusing the batch-size-64 reference.
Calculate and report optimizer updates, plot the required curves,
analyze convergence and variability, and propose one controlled
follow-up diagnosis without running it.

In [47]:
# Your code

**Batch-size diagnosis:**

## 2.7 Compare regularization methods

Compare the unregularized reference with the required L2 and dropout
configurations. Apply L2 to the kernel of each hidden `Dense` layer and
add dropout after each hidden layer. Report checkpoint training F1 and
validation metrics, quantify the training-minus-validation F1 gap, plot
comparable curves, and interpret their effects while accounting for the
meaning of the Keras loss and the training-only operation of dropout.

In [48]:
# Your code

**Regularization analysis:**

## Optional: Combined L2 and dropout

If attempted, train the four combined configurations in the handout.
Reuse the required reference runs, present the controlled comparisons,
and determine whether the methods appear complementary, redundant, or
too strong when combined.

In [49]:
# Optional code

**Optional combined-regularization analysis:**

## 2.8 Select, refit, and test one final model

Present the compact comparison table and select one configuration using
validation evidence. State and lock the complete preprocessing,
architecture, activation, regularization, optimizer, learning rate,
batch size, output, loss, seed, threshold, and checkpoint epoch before
accessing the test partition. Instantiate a fresh preprocessor, fit it
on the complete development partition, prepare fresh Keras arrays,
construct a fresh model, and train for the locked number of epochs.
Evaluate the model once on the common test partition. Compare it with
the saved test results for your identified Assignment 2 final model
without rerunning or reselecting that classical model, and do not revise
the neural model after seeing the test results.

In [50]:
# Your code

**Selection, test results, and interpretation:**

## 2.9 Produce predictions for independent hidden missions

Only after completing Task 8, train the locked configuration on all
labelled missions using a fresh preprocessor and a fresh model. The
supplied loading cell uses a local hidden file when available and
otherwise uses the course URL. Do not change the DataFrame name
`hidden_missions`.

In [51]:
hidden_source = HIDDEN_LOCAL_PATH if HIDDEN_LOCAL_PATH.exists() else HIDDEN_URL
hidden_missions = pd.read_csv(hidden_source)

print(f"Loaded hidden data from: {hidden_source}")
print(f"Hidden data: {hidden_missions.shape}")

Loaded hidden data from: https://raw.githubusercontent.com/turcotte/csi4106-f26/main/assignments-data/a2/asrc_missions_hidden.csv
Hidden data: (2124, 13)


Generate the hidden predictions and save
`asrc_missions_hidden_predictions.csv` with the exact columns and row
order specified in the handout. Do not round the probabilities before
saving them.

In [52]:
# Your code

Run the following structural validator. Retain its output in the
submitted notebook.

In [53]:
def validate_hidden_prediction_file(path, hidden_missions):
    expected_columns = [
        "mission_id",
        "predicted_class",
        "predicted_probability",
    ]
    predictions = pd.read_csv(path)

    if list(predictions.columns) != expected_columns:
        raise ValueError(
            f"Expected columns {expected_columns}, got {list(predictions.columns)}."
        )
    if len(predictions) != len(hidden_missions):
        raise ValueError(
            f"Expected {len(hidden_missions)} rows, got {len(predictions)}."
        )
    if predictions["mission_id"].duplicated().any():
        raise ValueError("mission_id values must be unique.")
    if not predictions["mission_id"].astype(str).equals(
        hidden_missions["mission_id"].astype(str)
    ):
        raise ValueError(
            "mission_id values and row order must match hidden_missions exactly."
        )

    predicted_class = pd.to_numeric(
        predictions["predicted_class"], errors="coerce"
    )
    if predicted_class.isna().any() or not set(predicted_class.unique()).issubset(
        {0, 1}
    ):
        raise ValueError("predicted_class must contain only 0 and 1.")

    predicted_probability = pd.to_numeric(
        predictions["predicted_probability"], errors="coerce"
    )
    if predicted_probability.isna().any():
        raise ValueError("predicted_probability must be complete and numeric.")
    if not predicted_probability.between(0, 1, inclusive="both").all():
        raise ValueError("predicted_probability must be between 0 and 1.")

    expected_class = (predicted_probability > 0.5).astype(int)
    submitted_class = predicted_class.astype(int)
    if not submitted_class.equals(expected_class):
        mismatches = int((submitted_class != expected_class).sum())
        raise ValueError(
            "predicted_class does not agree with predicted_probability "
            f"at threshold 0.5 for {mismatches} rows."
        )

    print(
        f"Valid prediction file: {path} "
        f"({len(predictions)} rows, structure and consistency verified)."
    )
    return predictions


validated_hidden_predictions = validate_hidden_prediction_file(
    "asrc_missions_hidden_predictions.csv",
    hidden_missions,
)

FileNotFoundError: [Errno 2] No such file or directory: 'asrc_missions_hidden_predictions.csv'

**Final hidden-prediction workflow:**

# References

List all external sources using the IEEE numbered reference style
described in the handout. Cite adapted code in a nearby comment or
Markdown cell. If you did not use external sources, state “No external
sources were used.” You remain responsible for verifying, understanding,
and explaining everything in your submission.

# AI Use Declaration

**1. Did you use a generative AI tool while completing this
assignment?** This includes assistance with code, debugging, analysis,
interpretation, writing, translation, or finding sources.

**Answer:** No / Yes

If you answered **No**, mark the confirmation below and omit questions 2
to 6 and the AI Interaction Transcript.

- [ ] I confirm that I did not use generative AI in preparing this
  submission.

If you answered **Yes**, complete questions 2 to 6.

**2. Which tools and models or versions did you use?**

**Answer:**

**3. How did you use generative AI?** Select all that apply.

- [ ] Clarifying concepts or checking my understanding
- [ ] Brainstorming or planning an approach
- [ ] Writing or revising code
- [ ] Debugging code or interpreting error messages
- [ ] Interpreting data, results, tables, or plots
- [ ] Writing or revising explanations
- [ ] Translation
- [ ] Finding references or documentation
- [ ] Other:

**4. Briefly identify the sections or tasks for which AI assistance was
used and describe how it influenced your work.**

**Answer:**

**5. How did you verify the AI-generated suggestions, code, or
explanations?**

**Answer:**

**6. Confirmation**

- [ ] I confirm that this declaration is complete, that the required
  transcript is included below, and that I understand and can explain
  everything in my submission.

# AI Interaction Transcript

If you used AI assistance, identify each conversation or session in the
numbered references and include the complete transcript here. If you did
not use AI assistance, omit this section.